# 🪡 Needle In A Haystack — LLM 长上下文检索能力评测

## 实验概述

**"大海捞针"（Needle In A Haystack, NIAH）** 是目前评估大模型长上下文能力最经典的压力测试。

### 核心思想
1. 构造一段**很长的无关文本**（haystack / 干草堆）
2. 在文本的**某个深度位置**插入一条关键事实（needle / 针）
3. 在文本末尾向模型提问，看它能否**准确检索**到那条事实
4. 遍历**不同上下文长度 × 不同插入深度**，绘制热力图

### 评测维度
- **X 轴**: 上下文总长度（如 1K → 128K tokens）
- **Y 轴**: 针的插入深度（0% = 开头, 100% = 结尾）
- **颜色**: 检索准确率（绿 = 成功, 红 = 失败）

### 本 Notebook 包含
| 模块 | 说明 |
|------|------|
| **Part 1** | 评测框架核心代码 |
| **Part 2** | 模拟模式 — 基于典型 attention 衰减模型生成仿真结果 |
| **Part 3** | 经典热力图可视化 |
| **Part 4** | 失败模式深度分析 |
| **Part 5** | 真实 API 模式 — 接入 OpenAI / vLLM / Ollama |
| **Part 6** | MLA vs MHA 长上下文保真度对比 |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import json, os, time, re
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Callable

# 设置中文字体和风格
plt.rcParams['font.family'] = ['Arial Unicode MS', 'SimHei', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False
plt.style.use('seaborn-v0_8-whitegrid')

print("依赖加载完成 ✅")

## Part 1: 评测框架核心

In [ ]:
@dataclass
class NeedleConfig:
    """大海捞针评测配置"""
    # --- 针 (Needle) ---
    needle: str = "The best thing to do in San Francisco is eat a sandwich and sit in Dolores Park on a sunny day."
    question: str = "What is the best thing to do in San Francisco?"
    expected_answer: str = "eat a sandwich and sit in Dolores Park"
    
    # --- 评测范围 ---
    context_lengths: List[int] = field(default_factory=lambda: [
        1000, 2000, 4000, 8000, 16000, 32000, 64000, 128000
    ])
    depth_percents: List[float] = field(default_factory=lambda: [
        0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100
    ])
    scoring_method: str = "substring"


class HaystackGenerator:
    """干草堆文本生成器"""
    
    FILLER_PARAGRAPHS = [
        "The rapid advancement of artificial intelligence has transformed various industries, from healthcare to finance. Machine learning algorithms now process vast amounts of data to identify patterns that were previously invisible to human analysts. Deep learning, in particular, has shown remarkable progress in natural language processing, computer vision, and reinforcement learning tasks.",
        "Climate change continues to be one of the most pressing challenges facing humanity. Rising global temperatures have led to more frequent extreme weather events, melting ice caps, and rising sea levels. Scientists around the world are working on innovative solutions, including carbon capture technologies and renewable energy systems.",
        "The history of computing is a fascinating journey from mechanical calculators to quantum processors. Charles Babbage conceived the first general-purpose computer in the 1830s, while Alan Turing laid the theoretical groundwork for modern computing in the 1930s. The transistor revolution of the 1950s paved the way for the personal computing revolution.",
        "Ocean exploration remains one of the final frontiers of scientific discovery. More than 80 percent of the ocean floor remains unmapped and unexplored. Marine biologists continue to discover new species in the deep sea, while oceanographers study the complex currents that regulate global climate patterns.",
        "The development of space exploration technology has accelerated dramatically in recent years. Private companies like SpaceX and Blue Origin have made significant strides in reducing the cost of access to space. Plans for lunar bases and Mars missions are active areas of research and development.",
        "Modern architecture has evolved to embrace sustainability and environmental consciousness. Green buildings incorporate solar panels, rainwater harvesting systems, and natural ventilation to minimize their ecological footprint. Architects are increasingly using recycled materials and biomimicry principles.",
        "The field of genetics has undergone revolutionary changes since the completion of the Human Genome Project. CRISPR-Cas9 gene editing technology has opened up new possibilities for treating genetic diseases. Personalized medicine promises to transform healthcare by tailoring treatments to individual genetic profiles.",
        "Mathematics serves as the universal language of science, providing the tools to describe and predict natural phenomena. From the elegance of Euler's identity to the complexity of the Riemann hypothesis, mathematical discoveries continue to push the boundaries of human understanding.",
        "The global food system faces numerous challenges, including population growth, resource scarcity, and changing dietary preferences. Vertical farming, precision agriculture, and lab-grown meat represent innovative approaches to feeding the world's growing population while minimizing environmental impact.",
        "Neuroscience research has made remarkable progress in understanding the human brain. Advanced neuroimaging techniques allow scientists to observe brain activity in real-time, revealing the neural correlates of consciousness, memory, and emotion.",
        "The evolution of transportation technology reflects humanity's constant desire for faster and more efficient mobility. From horse-drawn carriages to electric vehicles, each era has brought transformative changes. Autonomous driving and hyperloop systems promise to reshape mobility.",
        "Renewable energy technologies have reached a tipping point, with solar and wind power now cheaper than fossil fuels in many regions. Battery storage solutions are advancing rapidly, addressing the intermittency challenges associated with renewable sources.",
    ]
    
    @staticmethod
    def estimate_tokens(text: str) -> int:
        """粗略估算 token 数 (英文 ~1.3 tokens/word)"""
        return int(len(text.split()) * 1.3)
    
    @classmethod
    def generate_haystack(cls, target_tokens: int) -> str:
        """生成指定 token 数的干草堆文本"""
        paragraphs = []
        current_tokens = 0
        idx = 0
        while current_tokens < target_tokens:
            p = cls.FILLER_PARAGRAPHS[idx % len(cls.FILLER_PARAGRAPHS)]
            paragraphs.append(p)
            current_tokens += cls.estimate_tokens(p)
            idx += 1
        return "\n\n".join(paragraphs)
    
    @classmethod
    def insert_needle(cls, haystack: str, needle: str, depth_percent: float) -> str:
        """在干草堆的指定深度插入针"""
        paragraphs = haystack.split("\n\n")
        n = len(paragraphs)
        insert_idx = max(0, min(n, int(n * depth_percent / 100)))
        paragraphs.insert(insert_idx, needle)
        return "\n\n".join(paragraphs)
    
    @classmethod
    def build_prompt(cls, context: str, question: str) -> str:
        """构造最终 prompt"""
        return (
            f"<context>\n{context}\n</context>\n\n"
            f"Based on the context above, answer the following question. "
            f"Be concise and specific.\n\n"
            f"Question: {question}\nAnswer:"
        )


class NeedleScorer:
    """评分器"""
    
    @staticmethod
    def substring_match(response: str, expected: str) -> float:
        """子串匹配评分 (0-10)"""
        response_lower = response.lower().strip()
        expected_lower = expected.lower().strip()
        
        if expected_lower in response_lower:
            return 10.0
        
        keywords = expected_lower.split()
        matched = sum(1 for kw in keywords if kw in response_lower)
        ratio = matched / len(keywords)
        
        if ratio >= 0.8: return 8.0
        elif ratio >= 0.5: return 5.0
        elif ratio >= 0.2: return 2.0
        else: return 0.0


# --- 快速验证 ---
config = NeedleConfig()
gen = HaystackGenerator()

haystack = gen.generate_haystack(target_tokens=4000)
print(f"干草堆: ~{gen.estimate_tokens(haystack)} tokens, {len(haystack)} chars")

context = gen.insert_needle(haystack, config.needle, depth_percent=50)
prompt = gen.build_prompt(context, config.question)
print(f"插入针后: ~{gen.estimate_tokens(context)} tokens")
print(f"\nPrompt 头部:\n{prompt[:200]}...")
print(f"\nPrompt 尾部:\n...{prompt[-200:]}")

## Part 2: 模拟模式 — 基于 Attention 衰减理论的仿真

在没有 API 的情况下，基于各模型的 attention 衰减特征模拟检索结果：

| 模型类型 | Attention 衰减特征 | 预期表现 |
|---------|-------------------|----------|
| GPT-4 (128K) | 极轻微衰减 + Attention Sink | 几乎全绿 |
| Claude-3.5 (200K) | 均匀注意力 | 全绿 |
| Llama-3 (8K→128K) | 明显 "Lost in the Middle" | 两端强、中间弱 |
| 开源 7B (4K) | 超出窗口后急剧衰减 | 短绿长红 |
| DeepSeek-V3 MLA (128K) | 压缩不影响检索 | 类似 GPT-4 |

In [ ]:
class AttentionDecaySimulator:
    """
    基于注意力衰减理论模拟不同模型的长上下文检索能力。
    核心: needle 位置是否落在"有效注意力窗口"内 → 决定检索成功率
    """
    
    @staticmethod
    def _add_noise(score: float, noise_std: float = 0.05) -> float:
        noisy = score + np.random.normal(0, noise_std)
        return np.clip(noisy, 0, 10)
    
    @staticmethod
    def gpt4_128k(context_length: int, depth_percent: float) -> float:
        """GPT-4 Turbo (128K): 极强长上下文, 轻微中间遗忘"""
        base = 9.5
        len_pen = max(0, (context_length - 64000) / 128000) * 1.5
        mid_d = abs(depth_percent - 50) / 50
        mid_pen = (1 - mid_d) * 0.8 * (context_length / 128000)
        return AttentionDecaySimulator._add_noise(max(base - len_pen - mid_pen, 2))
    
    @staticmethod
    def claude3_200k(context_length: int, depth_percent: float) -> float:
        """Claude-3.5 (200K): 几乎完美均匀注意力"""
        base = 9.8
        len_pen = max(0, (context_length - 100000) / 200000) * 1.0
        mid_d = abs(depth_percent - 50) / 50
        mid_pen = (1 - mid_d) * 0.3 * (context_length / 200000)
        return AttentionDecaySimulator._add_noise(max(base - len_pen - mid_pen, 5))
    
    @staticmethod
    def llama3_128k(context_length: int, depth_percent: float) -> float:
        """Llama-3 (8K→128K RoPE extension): 明显 Lost in the Middle"""
        base = 9.0
        len_pen = min(3, max(0, (context_length - 8000) / 32000 * 2.5)) if context_length > 8000 else 0
        mid_d = abs(depth_percent - 50) / 50
        mid_pen = (1 - mid_d) ** 1.5 * 3.0 * min(1, context_length / 32000)
        sink = 1.5 if depth_percent < 10 else 0
        return AttentionDecaySimulator._add_noise(max(base - len_pen - mid_pen + sink, 0.5))
    
    @staticmethod
    def open_7b_4k(context_length: int, depth_percent: float) -> float:
        """开源 7B (4K 窗口): 超出窗口急剧失效"""
        native = 4096
        if context_length <= native:
            base = 8.5
            mid_pen = (1 - abs(depth_percent - 50) / 50) * 1.0
            return AttentionDecaySimulator._add_noise(max(base - mid_pen, 3))
        else:
            needle_pos = context_length * depth_percent / 100
            visible_start = context_length - native
            if needle_pos >= visible_start:
                return AttentionDecaySimulator._add_noise(7.0)
            elif needle_pos < 200:  # Attention Sink
                return AttentionDecaySimulator._add_noise(4.0)
            else:
                decay = max(0, 1 - (visible_start - needle_pos) / native)
                return AttentionDecaySimulator._add_noise(max(decay * 3, 0))
    
    @staticmethod
    def deepseek_mla_128k(context_length: int, depth_percent: float) -> float:
        """DeepSeek-V3 MLA (128K): MLA 压缩不影响检索"""
        base = 9.3
        len_pen = max(0, (context_length - 64000) / 128000) * 1.2
        mid_d = abs(depth_percent - 50) / 50
        mid_pen = (1 - mid_d) * 0.6 * (context_length / 128000)
        return AttentionDecaySimulator._add_noise(max(base - len_pen - mid_pen, 3))


# --- 运行全部 5 个模型模拟 ---
context_lengths = [1000, 2000, 4000, 8000, 16000, 32000, 64000, 128000]
depth_percents = list(range(0, 101, 5))  # 0%→100%, 步长 5%

models = {
    'GPT-4 Turbo (128K)': AttentionDecaySimulator.gpt4_128k,
    'Claude-3.5 Sonnet (200K)': AttentionDecaySimulator.claude3_200k,
    'Llama-3 (8K→128K RoPE)': AttentionDecaySimulator.llama3_128k,
    'Open-Source 7B (4K)': AttentionDecaySimulator.open_7b_4k,
    'DeepSeek-V3 MLA (128K)': AttentionDecaySimulator.deepseek_mla_128k,
}

np.random.seed(42)
all_results = {}
for name, fn in models.items():
    res = np.zeros((len(depth_percents), len(context_lengths)))
    for i, d in enumerate(depth_percents):
        for j, cl in enumerate(context_lengths):
            res[i, j] = fn(cl, d)
    all_results[name] = res
    print(f"  {name:35s} | 均分: {res.mean():.2f}/10 | min: {res.min():.1f} | max: {res.max():.1f}")

print(f"\n评测矩阵: {len(depth_percents)} depths × {len(context_lengths)} lengths = {len(depth_percents)*len(context_lengths)} cells")

## Part 3: 经典热力图可视化

大海捞针评测最标志性的可视化 — **热力图 (Heatmap)**
- 🟢 **绿色**: 满分通过  | 🟡 **黄色**: 部分正确 | 🔴 **红色**: 完全失败

In [ ]:
def plot_needle_heatmap(results, context_lengths, depth_percents,
                        title="Needle In A Haystack", ax=None, show_values=False):
    """绘制大海捞针经典热力图"""
    if ax is None:
        fig, ax = plt.subplots(figsize=(14, 8))
    
    # 自定义色图: 红→黄→绿
    cmap = mcolors.LinearSegmentedColormap.from_list('niah', [
        (0.0, '#d32f2f'), (0.2, '#f44336'), (0.4, '#ff9800'),
        (0.5, '#ffc107'), (0.7, '#8bc34a'), (0.8, '#4caf50'), (1.0, '#1b5e20'),
    ])
    
    im = ax.imshow(results, cmap=cmap, aspect='auto', vmin=0, vmax=10, interpolation='nearest')
    
    # X轴
    x_labels = [f"{cl//1000}K" if cl >= 1000 else str(cl) for cl in context_lengths]
    ax.set_xticks(range(len(context_lengths)))
    ax.set_xticklabels(x_labels, fontsize=10)
    ax.set_xlabel('Context Length (tokens)', fontsize=12, fontweight='bold')
    
    # Y轴
    y_step = max(1, len(depth_percents) // 10)
    ax.set_yticks(range(0, len(depth_percents), y_step))
    ax.set_yticklabels([f"{depth_percents[i]}%" for i in range(0, len(depth_percents), y_step)], fontsize=10)
    ax.set_ylabel('Needle Depth (0%=top, 100%=bottom)', fontsize=12, fontweight='bold')
    
    avg = results.mean()
    ax.set_title(f'{title}\nAvg Score: {avg:.1f}/10', fontsize=13, fontweight='bold', pad=12)
    
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Score (0-10)', fontsize=11)
    cbar.set_ticks([0, 2, 4, 6, 8, 10])
    cbar.set_ticklabels(['0\n(Failed)', '2', '4', '6', '8', '10\n(Perfect)'])
    
    if show_values and results.size <= 200:
        for i in range(results.shape[0]):
            for j in range(results.shape[1]):
                v = results[i, j]
                ax.text(j, i, f'{v:.0f}', ha='center', va='center',
                       fontsize=7, color='white' if v < 4 else 'black', fontweight='bold')
    return ax


# === 单模型热力图: GPT-4 ===
fig, ax = plt.subplots(figsize=(14, 8))
plot_needle_heatmap(all_results['GPT-4 Turbo (128K)'], context_lengths, depth_percents,
                    title='GPT-4 Turbo (128K) — Needle In A Haystack', ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# === 🔥 核心输出: 5 模型对比热力图 ===

fig, axes = plt.subplots(2, 3, figsize=(24, 14))
axes_flat = axes.flatten()

for idx, (name, results) in enumerate(all_results.items()):
    plot_needle_heatmap(results, context_lengths, depth_percents, title=name, ax=axes_flat[idx])

# 第6格: 统计摘要
ax_s = axes_flat[-1]
ax_s.axis('off')
summary = "📊 Model Comparison\n" + "="*32 + "\n\n"
for name, results in all_results.items():
    avg = results.mean()
    short = name.split('(')[0].strip()
    bar = '█' * int(avg) + '░' * (10 - int(avg))
    summary += f"{short:20s}\n  {bar} {avg:.1f}/10\n\n"
ax_s.text(0.5, 0.5, summary, transform=ax_s.transAxes, fontsize=11, fontfamily='monospace',
          va='center', ha='center', bbox=dict(boxstyle='round,pad=0.8', facecolor='#f5f5f5', edgecolor='#ddd'))

fig.suptitle('🪡 Needle In A Haystack — Multi-Model Comparison', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Part 4: 失败模式深度分析

In [ ]:
# === 失败模式三维分析 ===

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
colors_line = ['#e74c3c', '#2196f3', '#ff9800', '#9e9e9e', '#4caf50']

# (a) Score vs Context Length
ax = axes[0]
for (name, results), color in zip(all_results.items(), colors_line):
    short = name.split('(')[0].strip()
    ax.plot(context_lengths, results.mean(axis=0), 'o-', lw=2, ms=6, color=color, label=short)
ax.set_xlabel('Context Length', fontweight='bold'); ax.set_ylabel('Avg Score', fontweight='bold')
ax.set_title('(a) Score vs Context Length', fontweight='bold')
ax.set_xscale('log', base=2); ax.legend(fontsize=8, loc='lower left'); ax.set_ylim(0, 11)
ax.axhline(y=8, color='gray', ls='--', alpha=0.5)

# (b) Score vs Needle Depth — "Lost in the Middle" 诊断
ax = axes[1]
for (name, results), color in zip(all_results.items(), colors_line):
    short = name.split('(')[0].strip()
    ax.plot(depth_percents, results.mean(axis=1), '-', lw=2, color=color, label=short)
ax.set_xlabel('Needle Depth (%)', fontweight='bold'); ax.set_ylabel('Avg Score', fontweight='bold')
ax.set_title('(b) "Lost in the Middle" Diagnostic', fontweight='bold')
ax.legend(fontsize=8); ax.set_ylim(0, 11)
ax.axvspan(30, 70, alpha=0.1, color='red')
ax.axhline(y=8, color='gray', ls='--', alpha=0.5)

# (c) 通过率柱状图
ax = axes[2]
model_names = [n.split('(')[0].strip() for n in all_results.keys()]
pass_rates = [(r >= 8).mean() * 100 for r in all_results.values()]
avg_scores = [r.mean() for r in all_results.values()]
bars = ax.barh(model_names, pass_rates, color=colors_line, edgecolor='black', lw=0.5)
for bar, rate, avg in zip(bars, pass_rates, avg_scores):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{rate:.0f}% (avg {avg:.1f})', va='center', fontsize=10, fontweight='bold')
ax.set_xlabel('Pass Rate (score ≥ 8)', fontweight='bold')
ax.set_title('(c) Overall Pass Rate', fontweight='bold')
ax.set_xlim(0, 115)
ax.axvline(x=90, color='green', ls=':', lw=2, alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# === "Lost in the Middle" 现象 + Attention 分布图解 ===

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# (a) Llama-3 不同长度下的深度曲线
ax = axes[0]
llama_res = all_results['Llama-3 (8K→128K RoPE)']
colors_ctx = plt.cm.viridis(np.linspace(0, 1, len(context_lengths)))
for j, (cl, c) in enumerate(zip(context_lengths, colors_ctx)):
    label = f'{cl//1000}K' if cl >= 1000 else str(cl)
    ax.plot(depth_percents, llama_res[:, j], '-', lw=1.5, color=c, label=label, alpha=0.8)
ax.set_xlabel('Needle Depth (%)', fontweight='bold')
ax.set_ylabel('Score', fontweight='bold')
ax.set_title('Llama-3: "Lost in the Middle" at Different Lengths', fontweight='bold')
ax.legend(title='Context', fontsize=7, ncol=2)
ax.axvspan(30, 70, alpha=0.08, color='red')
ax.text(50, 0.5, '← Danger Zone →', ha='center', fontsize=10, color='red', alpha=0.5)
ax.set_ylim(-0.5, 11)

# (b) Attention 衰减机制图解
ax = axes[1]
positions = np.linspace(0, 1, 200)
sink = 3 * np.exp(-positions * 20) + 0.2
recency = 2 * np.exp(-(1 - positions) * 10) + 0.3
actual = sink + recency + 0.3
actual = actual / actual.sum() * len(positions)

ax.fill_between(positions * 100, 0, sink / sink.max(), alpha=0.3, color='#e74c3c', label='Attention Sink (开头)')
ax.fill_between(positions * 100, 0, recency / recency.max(), alpha=0.3, color='#2196f3', label='Recency Bias (结尾)')
ax.plot(positions * 100, actual / actual.max(), 'k-', lw=2, label='Actual (合成)')
ax.axhline(y=0.3 / actual.max(), color='green', ls='--', lw=1.5, label='Ideal Uniform')
ax.set_xlabel('Position in Context (%)', fontweight='bold')
ax.set_ylabel('Relative Attention Weight', fontweight='bold')
ax.set_title('Why "Lost in the Middle" Happens\n(Attention Distribution Anatomy)', fontweight='bold')
ax.legend(fontsize=9); ax.set_xlim(0, 100)
ax.annotate('← 高检索率', xy=(5, 0.8), fontsize=10, color='red', fontweight='bold')
ax.annotate('低检索率 →', xy=(35, 0.15), fontsize=10, color='gray', fontweight='bold')
ax.annotate('← 高检索率', xy=(80, 0.55), fontsize=10, color='blue', fontweight='bold')

plt.tight_layout()
plt.show()

## Part 5: 真实 API 模式

将模拟替换为真实 LLM 调用。支持 **OpenAI / Anthropic / vLLM / Ollama**。

> ⚠️ 运行此部分需要配置 API Key 和网络访问

In [ ]:
class LLMBackend:
    """LLM 后端抽象层"""
    
    @staticmethod
    def call_openai(prompt, model="gpt-4o", temperature=0.0, max_tokens=100):
        """调用 OpenAI API (需 pip install openai; export OPENAI_API_KEY=...)"""
        try:
            from openai import OpenAI
            client = OpenAI()
            resp = client.chat.completions.create(
                model=model, messages=[{"role": "user", "content": prompt}],
                temperature=temperature, max_tokens=max_tokens)
            return resp.choices[0].message.content
        except Exception as e: return f"[ERROR] {e}"
    
    @staticmethod
    def call_anthropic(prompt, model="claude-3-5-sonnet-20241022", max_tokens=100):
        """调用 Anthropic API (需 pip install anthropic; export ANTHROPIC_API_KEY=...)"""
        try:
            import anthropic
            client = anthropic.Anthropic()
            resp = client.messages.create(model=model, max_tokens=max_tokens,
                                          messages=[{"role": "user", "content": prompt}])
            return resp.content[0].text
        except Exception as e: return f"[ERROR] {e}"
    
    @staticmethod
    def call_vllm(prompt, base_url="http://localhost:8000/v1", model="default", max_tokens=100):
        """调用本地 vLLM 服务"""
        try:
            from openai import OpenAI
            client = OpenAI(base_url=base_url, api_key="EMPTY")
            resp = client.chat.completions.create(
                model=model, messages=[{"role": "user", "content": prompt}],
                temperature=0.0, max_tokens=max_tokens)
            return resp.choices[0].message.content
        except Exception as e: return f"[ERROR] {e}"
    
    @staticmethod
    def call_ollama(prompt, model="llama3", base_url="http://localhost:11434"):
        """调用 Ollama 本地模型"""
        try:
            import requests
            resp = requests.post(f"{base_url}/api/generate",
                json={"model": model, "prompt": prompt, "stream": False,
                      "options": {"temperature": 0, "num_predict": 100}})
            return resp.json().get("response", "[ERROR]")
        except Exception as e: return f"[ERROR] {e}"


def run_real_evaluation(llm_fn, config, context_lengths=None, depth_percents=None, verbose=True):
    """使用真实 LLM 进行大海捞针评测"""
    if context_lengths is None: context_lengths = config.context_lengths
    if depth_percents is None: depth_percents = config.depth_percents
    
    gen = HaystackGenerator()
    scorer = NeedleScorer()
    n_d, n_l = len(depth_percents), len(context_lengths)
    results = np.zeros((n_d, n_l))
    logs = []
    total = n_d * n_l
    
    for i, depth in enumerate(depth_percents):
        for j, ctx_len in enumerate(context_lengths):
            haystack = gen.generate_haystack(ctx_len)
            context = gen.insert_needle(haystack, config.needle, depth)
            prompt = gen.build_prompt(context, config.question)
            
            t0 = time.time()
            response = llm_fn(prompt)
            latency = time.time() - t0
            
            score = scorer.substring_match(response, config.expected_answer)
            results[i, j] = score
            logs.append({'depth': depth, 'context_length': ctx_len, 'score': score,
                        'latency': latency, 'response': response[:200]})
            
            if verbose:
                status = '✅' if score >= 8 else '⚠️' if score >= 4 else '❌'
                count = i * n_l + j + 1
                print(f"  [{count:3d}/{total}] depth={depth:3.0f}% len={ctx_len:>6d} → score={score:.0f} {status} ({latency:.1f}s)")
    
    return results, logs


print("真实 API 框架加载完成 ✅")
print("\n使用示例:")
print("  # OpenAI")
print("  results, logs = run_real_evaluation(LLMBackend.call_openai, config)")
print("  # Ollama")
print("  results, logs = run_real_evaluation(")
print("      lambda p: LLMBackend.call_ollama(p, model='llama3'), config)")
print("  # vLLM")
print("  results, logs = run_real_evaluation(")
print("      lambda p: LLMBackend.call_vllm(p, model='Qwen/Qwen2-7B'), config)")

In [ ]:
# ================================================================
# 🚀 取消注释以下代码，接入真实 API 进行评测
# ================================================================

# --- 方式 1: OpenAI ---
# os.environ['OPENAI_API_KEY'] = 'sk-...'
# config_real = NeedleConfig(
#     context_lengths=[1000, 4000, 8000, 16000, 32000],
#     depth_percents=[0, 25, 50, 75, 100],
# )
# results_real, logs = run_real_evaluation(
#     lambda p: LLMBackend.call_openai(p, model='gpt-4o'), config_real,
#     context_lengths=config_real.context_lengths, depth_percents=config_real.depth_percents)
# fig, ax = plt.subplots(figsize=(12, 7))
# plot_needle_heatmap(results_real, config_real.context_lengths, config_real.depth_percents,
#                     title='GPT-4o — Real API', ax=ax, show_values=True)
# plt.tight_layout(); plt.show()

# --- 方式 2: Ollama (本地) ---
# results_ollama, logs_ollama = run_real_evaluation(
#     lambda p: LLMBackend.call_ollama(p, model='llama3'),
#     NeedleConfig(context_lengths=[1000, 2000, 4000], depth_percents=[0, 25, 50, 75, 100]))

print("💡 取消上方注释并配置 API Key 即可运行真实评测")
print("💡 推荐先用小规模 (5 depths × 3 lengths = 15 次调用) 测试")

## Part 6: MLA vs MHA — 长上下文保真度对比

**核心假设**: MLA 低秩压缩不破坏 needle 位置的注意力信号 → 检索能力不变

In [ ]:
import torch
import torch.nn.functional as F

def simulate_attention_retrieval(seq_len, needle_pos, d_head=128, n_heads=32,
                                  effective_rank=64, mode='mha'):
    """模拟 MHA vs MLA 对 needle 位置的注意力分配"""
    torch.manual_seed(42)
    q = torch.randn(n_heads, 1, d_head)
    
    if mode == 'mha':
        K = torch.randn(n_heads, seq_len, d_head) * 0.1
        K[:, needle_pos, :] += q.squeeze(1) * 0.5
    elif mode == 'mla':
        K_full = torch.randn(n_heads, seq_len, d_head) * 0.1
        K_full[:, needle_pos, :] += q.squeeze(1) * 0.5
        # SVD 截断模拟低秩压缩
        K_flat = K_full.reshape(n_heads * seq_len, d_head)
        U, S, Vh = torch.linalg.svd(K_flat, full_matrices=False)
        K_flat_c = U[:, :effective_rank] @ torch.diag(S[:effective_rank]) @ Vh[:effective_rank, :]
        K = K_flat_c.reshape(n_heads, seq_len, d_head)
    
    attn = F.softmax((q @ K.transpose(-2, -1)) * (d_head ** -0.5), dim=-1)
    needle_attn = attn[:, 0, needle_pos].mean().item()
    avg_dist = attn.mean(dim=0)[0]
    needle_rank = (avg_dist > avg_dist[needle_pos]).sum().item() + 1
    
    return {
        'needle_attn': needle_attn, 'needle_rank': needle_rank,
        'found': avg_dist.argmax().item() == needle_pos,
        'attn_dist': avg_dist.detach().numpy(),
    }


# --- 对比 MHA vs MLA ---
seq_lens_test = [256, 512, 1024, 2048, 4096]
print(f"{'Seq Len':>8} | {'Mode':>5} | {'Needle Attn':>12} | {'Rank':>6} | Found")
print("-" * 55)
for sl in seq_lens_test:
    for mode in ['mha', 'mla']:
        res = simulate_attention_retrieval(sl, sl // 2, mode=mode)
        s = '✅' if res['found'] else '❌'
        print(f"{sl:>8} | {mode:>5} | {res['needle_attn']:>12.6f} | {res['needle_rank']:>6d} | {s}")

In [ ]:
# === MHA vs MLA 注意力分布对比可视化 ===

fig, axes = plt.subplots(2, 3, figsize=(20, 10))
seq_vis = [256, 1024, 4096]

for j, sl in enumerate(seq_vis):
    needle_pos = sl // 2
    for i, mode in enumerate(['mha', 'mla']):
        ax = axes[i, j]
        res = simulate_attention_retrieval(sl, needle_pos, mode=mode)
        attn = res['attn_dist']
        
        color = '#2196f3' if mode == 'mha' else '#4caf50'
        ax.plot(range(sl), attn, '-', lw=0.5, alpha=0.7, color=color)
        ax.axvline(x=needle_pos, color='red', ls='--', lw=1.5, alpha=0.8, label=f'Needle @{needle_pos}')
        ax.scatter([needle_pos], [attn[needle_pos]], color='red', s=100, zorder=5, edgecolors='black', lw=1)
        
        label = 'Standard MHA' if mode == 'mha' else 'MLA (compressed)'
        status = '✅ Found' if res['found'] else f'❌ Rank {res["needle_rank"]}'
        ax.set_title(f'{label}, T={sl}\nRank: {res["needle_rank"]}, Attn: {res["needle_attn"]:.4f} {status}',
                     fontsize=10, fontweight='bold')
        ax.set_xlabel('Position'); ax.set_ylabel('Attention Weight'); ax.legend(fontsize=8)

axes[0, 0].set_ylabel('Standard MHA\nAttention Weight', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('MLA (Compressed)\nAttention Weight', fontsize=11, fontweight='bold')
fig.suptitle('MHA vs MLA: Needle Retrieval Attention Distribution', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 总结

### 关键发现

| 发现 | 说明 |
|------|------|
| **"Lost in the Middle"** | 多数模型在上下文中间段检索能力下降，两端表现更好 |
| **Attention Sink** | 开头 tokens 天然保留高注意力，形成"注意力沉没"现象 |
| **MLA 无损压缩** | MLA 低秩压缩 KV Cache 后，needle 位置的注意力信号几乎不受影响 |
| **窗口外失效** | 4K 窗口模型在超长上下文时，中间段信息完全丢失 |
| **RoPE 扩展的代价** | 从 8K 扩展到 128K 后，中间段检索能力明显下降 |

### 评测建议

1. **最小测试集**: 5 depths × 5 lengths = 25 次调用，快速诊断
2. **完整测试**: 21 depths × 8 lengths = 168 次调用，论文级热力图
3. **多 needle 变体**: 更换不同类型的事实（数字、人名、日期）
4. **多语言**: 同时使用中英文 needle，测试跨语言长上下文能力